In [ ]:
import os, sys
import pandas as pd
import numpy as np
import cv2
import openslide
from tqdm import tqdm

import PIL
from PIL import Image
from skimage import img_as_ubyte

from sklearn.cluster import KMeans
from collections import Counter

import pickle

import Tissue_Segmentation

In [ ]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

seed = 724
set_seed(seed)

In [ ]:
relevant_path = os.getcwd()

patch_Size_at_20x = 1024
patch_Size_at_5x = int(patch_Size_at_20x/4)

In [ ]:
def RGB2HSD(X): # Hue Saturation Density
    #
    # Function to convert RGB to HSD
    # from https://github.com/FarhadZanjani/Histopathology-Stain-Color-Normalization/blob/master/ops.py
    # Args:
    #     X: RGB image
    # Returns:
    #     X_HSD: HSD image
    #
    eps = np.finfo(float).eps # Epsilon
    X[np.where(X==0.0)] = eps # Changing zeros with epsilon
    OD = -np.log(X / 1.0) # It seems to be calculating the Optical Density
    D  = np.mean(OD,3) # Getting density?
    D[np.where(D==0.0)] = eps # Changing zero densitites with epsilon
    cx = OD[:,:,:,0] / (D) - 1.0 
    cy = (OD[:,:,:,1]-OD[:,:,:,2]) / (np.sqrt(3.0)*D)
    D = np.expand_dims(D,3) # Hue?
    cx = np.expand_dims(cx,3) # Saturation
    cy = np.expand_dims(cy,3) # Density?
    X_HSD = np.concatenate((D,cx,cy),3)
    return X_HSD

def clean_thumbnail(thumbnail):
    # 
    # Function to clean thumbnail
    # Args:
    #     thumbnail: thumbnail image
    # Returns:
    #     wthumbnail: cleaned thumbnail image
    # 
    # thumbnail array
    thumbnail_arr = np.asarray(thumbnail)
    # writable thumbnail
    wthumbnail = np.zeros_like(thumbnail_arr)
    wthumbnail[:, :, :] = thumbnail_arr[:, :, :]
    # Remove pen marking here
    # We are skipping this
    # This  section sets regoins with white spectrum as the backgroud regoin
    thumbnail_std = np.std(wthumbnail, axis=2)
    wthumbnail[thumbnail_std<5] = (np.ones((1,3), dtype="uint8")*255)
    thumbnail_HSD = RGB2HSD(np.array([wthumbnail.astype('float32')/255.]))[0]
    kernel = np.ones((30,30),np.float32)/900
    thumbnail_HSD_mean = cv2.filter2D(thumbnail_HSD[:,:,2],-1,kernel)
    wthumbnail[thumbnail_HSD_mean<0.05] = (np.ones((1,3),dtype="uint8")*255)
    # return writable thumbnail
    return wthumbnail


def get_patches(slide, tissue_mask):
    # 
    # Function to get patches
    # Args:
    #     slide: slide object
    #     tissue_mask: tissue mask
    # Returns:
    #     patches: list of patches
    # 
    # getting slide dimensions and objective power
    w, h = slide.dimensions
    try:
        objective_power = int(slide.properties['openslide.objective-power'])
    except Exception as e:
        objective_power = 40.0
    print("Objective Power: ", objective_power)
    # at 20x its 1024x1024
    patch_size = (objective_power/20.)*patch_Size_at_20x
    # getting mask ratios
    mask_hratio = (tissue_mask.shape[0]/h)*patch_size
    mask_wratio = (tissue_mask.shape[1]/w)*patch_size
    
    # iterating over patches
    patches = []
    for i, hi in enumerate(range(0, h, int(patch_size))):
        # iterating over patches
        _patches = []
        for j, wi in enumerate(range(0, w, int(patch_size))):
            # check if patch contains 70% tissue area
            mi = int(i*mask_hratio)
            mj = int(j*mask_wratio)
            # get patch mask
            patch_mask = tissue_mask[mi:mi+int(mask_hratio), mj:mj+int(mask_wratio)]
            # get tissue coverage
            tissue_coverage = np.count_nonzero(patch_mask)/patch_mask.size
            # Add patch to list
            _patches.append({'loc': [i, j], 'wsi_loc': [int(hi), int(wi)], 'tissue_coverage': tissue_coverage})
        # Add patches to list
        patches.append(_patches)
    # return patches
    return patches
        

def get_flat_pathces(slide, patches, tissue_threshold):  
    # 
    # Function to get flat patches
    # Args:
    #     slide: slide object
    #     patches: list of patches
    #     tissue_threshold: threshold for tissue coverage
    # Returns:
    #     flat_patches: list of flat patches
    # 
    # Converting patches to flat patches
    flat_patches = np.ravel(patches)
    # Getting the objective power
    try:
        objective_power = int(slide.properties['openslide.objective-power'])
    except Exception as e:
        objective_power = 40.0

    # Iterating over patches
    for patch in tqdm(flat_patches):
        # ignore patches with less tissue coverage
        if patch['tissue_coverage'] < tissue_threshold:
            continue
        # this loc is at the objective power
        h, w = patch['wsi_loc']
        # we will go one level lower, i.e. (objective power / 4)
        # we still need patches at 5x of size 256x256
        # this logic can be modified and may not work properly for images of lower objective power < 20 or greater than 40
        patch_size_5x = int(((objective_power / 4)/5)*patch_Size_at_5x)
        # read the patch
        patch_region = slide.read_region((w, h), 1, (patch_size_5x, patch_size_5x)).convert('RGB')
        # resize to 256x256
        if patch_region.size[0] != patch_Size_at_5x:
            patch_region = patch_region.resize((patch_Size_at_5x, patch_Size_at_5x))
        # convert to numpy array
        histogram = (np.array(patch_region)/255.).reshape((patch_Size_at_5x*patch_Size_at_5x, 3)).mean(axis=0)
        patch['rgb_histogram'] = histogram  
    # return flat patches
    return flat_patches
        
def get_mosaic(flat_patches, kmeans_clusters, tissue_threshold, percentage_selected):
    # 
    # Function to get mosaic from flat patches
    # Args:
    #     flat_patches: list of patches
    #     kmeans_clusters: number of clusters for kmeans
    #     tissue_threshold: threshold for tissue coverage
    #     percentage_selected: percentage of patches to be selected from each cluster
    # Returns:
    #     mosaic: list of patches to be used for mosaic
    # 
    # select patches with tissue coverage greater than threshold
    selected_patches_flags = [patch['tissue_coverage'] >= tissue_threshold for patch in flat_patches]
    selected_patches = flat_patches[selected_patches_flags]
    # Finding the number of patches as minimum of set kmeans_clusters and number of selected patches
    kmeans_clusters = min(kmeans_clusters, len(selected_patches))
    # run kmeans on rgb histogram
    kmeans = KMeans(n_clusters = kmeans_clusters, random_state=724)
    # get rgb histogram
    features = np.array([entry['rgb_histogram'] for entry in selected_patches])
    # fit kmeans
    kmeans.fit(features)
    # initialize mosaic
    mosaic = []
    # iterate over clusters
    for i in range(kmeans_clusters):
        # select patches from each cluster
        cluster_patches = selected_patches[kmeans.labels_ == i]
        # find number of patches to be selected from this cluster
        n_selected = max(1, int(len(cluster_patches)*percentage_selected/100.))
        # initialize kmeans
        km = KMeans(n_clusters=n_selected, random_state=724)
        # get location features
        loc_features = [patch['wsi_loc'] for patch in cluster_patches]
        # fit kmeans
        ds = km.fit_transform(loc_features)
        # initialize selected idx
        c_selected_idx = []
        # iterate over selected patches
        for idx in range(n_selected):
            # sort idx based on distance
            sorted_idx = np.argsort(ds[:, idx])
            # iterate over sorted idx
            for sidx in sorted_idx:
                if sidx not in c_selected_idx:
                    # add idx to selected idx if not already selected
                    c_selected_idx.append(sidx)
                    mosaic.append(cluster_patches[sidx])
                    break
    # return mosaic       
    return mosaic

In [ ]:
def draw_mosaic_on_thumbnail(slide, thumbnail, mosaic):

    w, h = slide.dimensions

    t_w, t_h = [thumbnail.shape[1], thumbnail.shape[0]]
    try:
        objective_power = int(slide.properties['openslide.objective-power'])
    except Exception as e:
        objective_power = 40.0
    # at 20x its 1024x1024
    patch_size = (objective_power/20.)*patch_Size_at_20x
    # getting mask ratios
    mask_hratio = int(h/t_h)
    mask_wratio = int(w/t_w)

    smaller_patch_size = int(patch_size/mask_hratio)

    Img = np.array(thumbnail)
    for patch_id, patch_coord in enumerate(mosaic):
        patch_coord = patch_coord["wsi_loc"]
        x, y = patch_coord[0], patch_coord[1]
        x = int(x/mask_wratio)
        y = int(y/mask_hratio)
        Img = cv2.rectangle(Img, (y, x), (y+smaller_patch_size, x+smaller_patch_size), (0, 0, 0), 2)
    
    return Img
    

In [ ]:
ScannerA_WSI_Path = "Path to the WSIs dir from Scanner A"
ScannerB_WSI_Path = "Path to the WSIs dir from Scanner B" ## optional for the second scanner

extensions = ['tif', 'tiff', 'svs']
WSI_List = [fn for fn in os.listdir(ScannerA_WSI_Path) if any(fn.endswith(ext) for ext in extensions)]

# Main Mosaic coords Generation Loop

In [ ]:
## Mosaic Controlled Parameter
percentage_selected = 5
tissue_threshold = 0.8
kmeans_clusters = 9

for W in WSI_List:
    W = W[:-4]
    print(W)

    ### check if Mosaic output dir already exist
    if os.path.exists(os.path.join(relevant_path, "ScannerA_HE_Mosaics")):
        print("Mosaic Path Exists")
    else:
        os.mkdir(os.path.join(relevant_path, "ScannerA_HE_Mosaics"))

    ### check if Mosaic already generated
    if os.path.exists(os.path.join(relevant_path, "ScannerA_HE_Mosaics", W+"_Mosaic.pkl")):
        print("Mosaic already Generated")
    else:

        slide_path = os.path.join(ScannerA_WSI_Path, W+".svs")

        # Create the slide object
        slide = openslide.open_slide(slide_path)
        # Get the thumbnail
        thumbnail, tissue_mask = Tissue_Segmentation.unet_tissue_segmentation(slide)  ## Generate Thumbnail and Mask (1024,1024)
        # Get the tissue mask
        cthumbnail = clean_thumbnail(thumbnail)

        # Get the patches
        patches = get_patches(slide, tissue_mask)
        flat_patches = get_flat_pathces(slide, patches, tissue_threshold)
        # Get the mosaic
        mosaic = get_mosaic(flat_patches, kmeans_clusters, tissue_threshold, percentage_selected)

        mosaic_img = draw_mosaic_on_thumbnail(slide, thumbnail, mosaic)
        mosaic_img = Image.fromarray(mosaic_img)
        mosaic_img.save(os.path.join(relevant_path, "ScannerA_HE_Mosaics", W+"_Mosaic.jpg"))

        ### Save Mosaic Dict as pkl dict
        with open(os.path.join(relevant_path, "ScannerA_HE_Mosaics", W+"_Mosaic.pkl"), 'wb') as f:
            pickle.dump(mosaic, f)

        ## Close WSI
        slide.close()
